# Preprocesamiento del dataset

In [3]:
# Instalamos Roboflow para poder descargar el dataset desde Roboflow Universe
!pip install ultralytics roboflow python-dotenv



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 124.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [4]:
# Importamos las librerías necesarias
import os
import sys
import shutil
import yaml

from roboflow import Roboflow
from google.colab import userdata

# Configurar y clonar la rama de trabajo de forma controlada
REPO_NAME = "tp_final_vpc_II_23co"
BRANCH_NAME = "eda"

if not os.path.exists(REPO_NAME):
    print(f"Clonando repositorio remoto (rama: {BRANCH_NAME})...")
    !git clone -b {BRANCH_NAME} https://github.com/PabloSalvagni/{REPO_NAME}.git
else:
    print("Sincronizando últimos cambios de Git...")
    %cd {REPO_NAME}
    !git pull origin {BRANCH_NAME}
    %cd ..

# Inyectamos el repositorio al path de Python para reconocer el módulo 'src'
repo_path = os.path.abspath(REPO_NAME)
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Posicionamos el directorio de trabajo en la raíz del proyecto
%cd {REPO_NAME}
print(f"Directorio de trabajo actual: {os.getcwd()}")

Clonando repositorio remoto (rama: eda)...
Cloning into 'tp_final_vpc_II_23co'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 39 (delta 7), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 9.88 MiB | 25.74 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/tp_final_vpc_II_23co
Directorio de trabajo actual: /content/tp_final_vpc_II_23co


In [5]:
# Descargamos el dataset original desde Roboflow en formato YOLOv8
api_key = userdata.get("RoboflowAPIKey")

rf = Roboflow(api_key=api_key)

project = rf.workspace("raiyan8018").project("sunflower-mn2cr")

dataset = project.version(1).download("yolov8")

original_dataset_path = dataset.location

print("Dataset original descargado en:")
print(original_dataset_path)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sunflower-1 in yolov8:: 100%|██████████| 8584/8584 [00:04<00:00, 2114.99it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset original descargado en:
/content/tp_final_vpc_II_23co/Sunflower-1


In [6]:
# Verificamos la estructura del dataset original
os.listdir(original_dataset_path)

['train',
 'data.yaml',
 'valid',
 'test',
 'README.roboflow.txt',
 'README.dataset.txt']

In [7]:
# Consultamos cuánto ocupa el dataset original antes de clonarlo
!du -sh {original_dataset_path}

360M	/content/tp_final_vpc_II_23co/Sunflower-1


### Clonación del dataset

In [8]:
# Creamos una copia del dataset original para no modificar los datos descargados desde Roboflow
unified_dataset_path = "/content/sunflower_unified"

if os.path.exists(unified_dataset_path):
    shutil.rmtree(unified_dataset_path)

shutil.copytree(original_dataset_path, unified_dataset_path)

print("Dataset clonado correctamente en:")
print(unified_dataset_path)

Dataset clonado correctamente en:
/content/sunflower_unified


In [9]:
# Verificamos que la copia tenga la misma estructura que el dataset original
os.listdir(unified_dataset_path)

['train',
 'data.yaml',
 'valid',
 'test',
 'README.roboflow.txt',
 'README.dataset.txt']

### Unificación de clases

In [10]:
# Unificamos las clases del dataset: cualquier class_id pasa a ser 0
for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        new_lines = []

        with open(label_path, "r") as file:
            lines = file.readlines()

        for line in lines:
            parts = line.strip().split()

            if len(parts) == 5:
                parts[0] = "0"
                new_lines.append(" ".join(parts))

        with open(label_path, "w") as file:
            file.write("\n".join(new_lines))

print("Clases unificadas correctamente.")

Clases unificadas correctamente.


In [11]:
# Actualizamos el archivo data.yaml para indicar que ahora existe una sola clase
yaml_path = os.path.join(unified_dataset_path, "data.yaml")

with open(yaml_path, "r") as file:
    data_yaml = yaml.safe_load(file)

data_yaml["nc"] = 1
data_yaml["names"] = ["sunflower"]

with open(yaml_path, "w") as file:
    yaml.dump(data_yaml, file, sort_keys=False)

print("data.yaml actualizado correctamente.")

data.yaml actualizado correctamente.


In [12]:
# Mostramos el contenido actualizado de data.yaml
with open(yaml_path, "r") as file:
    print(file.read())

names:
- sunflower
nc: 1
roboflow:
  license: CC BY 4.0
  project: sunflower-mn2cr
  url: https://universe.roboflow.com/raiyan8018/sunflower-mn2cr/dataset/1
  version: 1
  workspace: raiyan8018
test: ../test/images
train: ../train/images
val: ../valid/images



In [13]:
# Verificamos que todos los labels tengan únicamente class_id = 0
class_ids = set()

for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        with open(label_path, "r") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) == 5:
                    class_id = int(parts[0])
                    class_ids.add(class_id)

print("Clases presentes después de unificar:")
print(class_ids)

Clases presentes después de unificar:
{0}


In [14]:
# Verificamos nuevamente cantidad de imágenes y labels en cada partición del dataset unificado
for split in ["train", "valid", "test"]:
    images_dir = os.path.join(unified_dataset_path, split, "images")
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    n_images = len(os.listdir(images_dir))
    n_labels = len(os.listdir(labels_dir))

    print(f"{split}: {n_images} imágenes | {n_labels} labels")

train: 3429 imágenes | 3429 labels
valid: 428 imágenes | 428 labels
test: 429 imágenes | 429 labels


# Entrenamiento

In [15]:
import torch
from ultralytics import YOLO

# Verificacion de hardware para evitar ValueErrors
device_to_use = 0 if torch.cuda.is_available() else "cpu"
print(f"Dispositivo asignado para entrenamiento: {device_to_use}")

if device_to_use == "cpu":
    print("Corriendo en CPU.")

# Cargamos la arquitectura YOLOv8-Nano con pesos iniciales de COCO
model = YOLO("yolov8n.pt")

# Iniciamos el pipeline de fine-tuning sobre nuestra clase unificada
metrics = model.train(
    data=os.path.join(dataset.location, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device=device_to_use,
    workers=2,
    project="sunflower_counting",
    name="yolov8n_baseline",
    save=True,
    plots=True
)

print("Entrenamiento finalizado.")

Dispositivo asignado para entrenamiento: 0
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tp_final_vpc_II_23co/Sunflower-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_baseline, nbs=64, nms=False, ops